### 1. Carga de los conjuntos de datos

In [1]:
import pandas as pd 
import numpy as np

df_atletas = pd.read_csv('../data/athlete_events.csv')
df_regiones = pd.read_csv('../data/noc_regions.csv')

display(df_atletas.head(10))
display(df_regiones.head(10))

ModuleNotFoundError: No module named 'pandas'

### 2. Inspección general de estructura y valores nulos

In [2]:
print("Atletas:", df_atletas.shape)
print("Regiones:", df_regiones.shape)

df_atletas.info()
print("\nNulos en atletas:")
print(df_atletas.isnull().sum())

print("\nNulos en regiones:")
print(df_regiones.isnull().sum())

NameError: name 'df_atletas' is not defined

### Descripción de las variables de los datasets

#### 1. Dataset de eventos y atletas (`athlete_events.csv`)
Cada fila representa la participación de un deportista en una prueba específica dentro de una edición de los Juegos Olímpicos:

* **`ID`:** Identificador numérico único asignado a cada atleta.
* **`Name`:** Nombre completo del deportista.
* **`Sex`:** Sexo biológico registrado del atleta (`M` para masculino, `F` para femenino).
* **`Age`:** Edad del atleta (en años) al momento de competir en esa edición.
* **`Height`:** Estatura del deportista medida en centímetros (cm).
* **`Weight`:** Peso corporal del deportista medido en kilogramos (kg).
* **`Team`:** Nombre del equipo, delegación o país al que representó el atleta.
* **`NOC`:** Código de tres letras del Comité Olímpico Nacional.
* **`Games`:** Año y temporada de la edición de los Juegos Olímpicos.
* **`Year`:** Año calendario en que se celebraron los Juegos.
* **`Season`:** Temporada de los Juegos Olímpicos (`Summer` para Verano, `Winter` para Invierno).
* **`City`:** Ciudad anfitriona donde se realizó la competencia.
* **`Sport`:** Disciplina deportiva general.
* **`Event`:** Prueba o categoría específica dentro del deporte.
* **`Medal`:** Resultado obtenido en la prueba (`Gold`, `Silver`, `Bronze` o `NaN` si no ganó medalla).

---

#### 2. Dataset de regiones (`noc_regions.csv`)
Catálogo auxiliar que relaciona los códigos olímpicos con sus países o territorios actuales:

* **`NOC`:** Código de tres letras del Comité Olímpico Nacional (llave de unión con la tabla de atletas).
* **`region`:** Nombre del país o región geográfica actual correspondiente al código `NOC`.
* **`notes`:** Observaciones o aclaraciones históricas sobre códigos antiguos, territorios especiales o equipos independientes.

### 3. Estadísticas descriptivas iniciales

In [3]:
display(df_atletas.describe())
display(df_atletas.describe(include='object'))

NameError: name 'df_atletas' is not defined

### 4. Creación de la variable IMC

In [ ]:
df_atletas['IMC'] = df_atletas['Weight'] / ((df_atletas['Height'] / 100) ** 2)
display(df_atletas[['Name', 'Height', 'Weight', 'IMC']].dropna())

,Name,Height,Weight,IMC
0,A Dijiang,180.0,80.0,24.691358
1,A Lamusi,170.0,60.0,20.761246
4,Christine Jacoba Aaftink,185.0,82.0,23.959094
5,Christine Jacoba Aaftink,185.0,82.0,23.959094
6,Christine Jacoba Aaftink,185.0,82.0,23.959094
...,...,...,...,...
271111,Andrzej ya,179.0,89.0,27.776911
271112,Piotr ya,176.0,59.0,19.047004
271113,Piotr ya,176.0,59.0,19.047004
271114,Tomasz Ireneusz ya,185.0,96.0,28.049671


### Análisis de la variable IMC

* **¿Qué opina de esta nueva variable?**  
    Aunque el Índice de Masa Corporal (IMC) es una fórmula fácil y rápida para relacionar el peso con la estatura, no es la mejor medida para evaluar a deportistas de alto rendimiento. Como el cálculo solo toma en cuenta el peso total y no sabe diferenciar entre grasa y músculo, puede terminar clasificando a un atleta musculoso como si tuviera sobrepeso u obesidad, cuando en realidad tiene el físico ideal para su deporte.

* **¿Tiene el mismo sentido en todos los deportes?**  
  No, su interpretación cambia de forma importante según las exigencias físicas de cada disciplina:
  * **Deportes de fuerza** (como lanzamiento de bala o lucha): los atletas suelen presentar un IMC elevado, en la mayoria de los casos superior a 30, debido a la hipertrofia/desarrollo muscular necesario para competir, sin que esto represente un problema de salud.
  * **Deportes de resistencia o estética** (como maratón o gimnasia rítmica): predominan valores de IMC mucho más bajos, ya que un peso corporal ligero favorece la agilidad, la aerodinámica y la eficiencia energética.
  * **Conclusión:** El IMC no debe analizarse de forma general en toda la base de datos como un indicador de salud, sino compararse únicamente dentro de atletas de una misma disciplina o categoría.

### 5. Fusión (merge) de atletas con regiones

In [ ]:
df_atletas = pd.merge(df_atletas, df_regiones, on='NOC', how='left')

print("Filas antes del merge:", len(df_atletas))
print("Filas sin región asignada:", df_atletas['region'].isna().sum())

Filas antes del merge: 271116
Filas sin región asignada: 370


### Análisis de la fusión (Merge) de datos

* **Verificación de filas y registros sin región:**  
  Después de unir ambas tablas, el dataframe mantiene exactamente las mismas **271,116 filas** originales. Sin embargo, al revisar la columna `region`, encontramos **370 filas** que quedaron vacías (`NaN`).

* **¿Por qué ocurre esto?**  
  Esto pasa porque existen pequeñas diferencias entre los códigos olímpicos (`NOC`) registrados en la tabla de eventos y los de la tabla de regiones. El caso más claro es **Singapur**, que aparece bajo el código `SGP` en los eventos (349 filas), mientras que en el catálogo de regiones está guardado con su código antiguo (`SIN`). Los otros casos corresponden a delegaciones especiales o sin país fijo, como el Equipo Olímpico de Atletas Refugiados (`ROT`, 12 filas), Tuvalu (`TUV`, 7 filas) y atletas desconocidos (`UNK`, 2 filas).

* **¿Qué tipo de fusión (`how`) se utilizó y por qué?**  
  Se utilizó una fusión por la izquierda (`how='left'`). De esta manera nos aseguramos de conservar absolutamente todas las participaciones de los atletas (`df_atletas`), agregando la información geográfica cuando existe coincidencia y dejando vacío el campo cuando no la hay, en lugar de eliminar esos 370 registros como ocurriría con un `inner`.

### 6. Creación de la variable binaria medalla y comparación de registros

In [ ]:
df_atletas['medalla'] = df_atletas['Medal'].notna().astype(int)

print("Distribución de medallas (0 = Sin medalla, 1 = Con medalla):")
print(df_atletas['medalla'].value_counts())
print("\nTotal de filas en el dataset:", len(df_atletas))
print("Total de atletas únicos (ID):", df_atletas['ID'].nunique())

Distribución de medallas (0 = Sin medalla, 1 = Con medalla):
medalla
0    231333
1     39783
Name: count, dtype: int64

Total de filas en el dataset: 271116
Total de atletas únicos (ID): 135571


### Análisis de la variable medalla y registros únicos

* **¿Qué se hizo con los valores vacíos de `Medal`?**  
  En la base de datos original, la columna `Medal` solo registra texto cuando el deportista sube al podio (Gold, Silver o Bronze), dejando como NaN todas las demás participaciones. Por eso, esos valores vacíos no son errores de registro, sino participaciones sin medalla; al crear la variable binaria, se transformaron en **`0`** (231,333 registros) y los deportistas con medallas ganadas en **`1`** (39,783 registros).

* **Comparación entre filas y atletas únicos (`ID`):**  
  El dataset tiene **271,116 filas**, pero al revisar la columna `ID` vemos que solo hay **135,571 atletas únicos**. Esto ocurre porque un mismo deportista aparece varias veces si compitió en distintas pruebas o en diferentes ediciones de los Juegos.

* **¿Qué implica esto para el conteo de medallas?**  
  Implica que **sumar los `1` de la columna `medalla` no equivale al medallero oficial de los países**, especialmente en deportes de equipo (como fútbol, baloncesto o relevos). Si un equipo de 12 jugadores gana el oro, en la tabla aparecen 12 filas con `medalla = 1`, cuando en realidad para el medallero olímpico eso cuenta como una sola medalla para ese país.